In [1]:
# Install
!pip install -U scikit-image

/bin/bash: /home/jia/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 67.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scikit-image
    Found existing installation: scikit-image 0.23.2
    Uninstalling scikit-image-0.23.2:
      Successfully uninstalled scikit-image-0.23.2


In [2]:
#Import packages
import numpy as np
import pandas as pd
import os
import cv2
import skimage 
from glob import glob
import imutils
import matplotlib.pyplot as plt

from imutils import contours
from PIL import Image as Img
from IPython.display import Image

from joblib import Parallel, delayed
from tqdm import tqdm_notebook as tqdm
from tqdm.notebook import tqdm
from skimage import measure 
import shutil
import random

In [3]:
new_path = '/home/xin/Project/2024 full image dataset_3H7C_corrected'
new_filenames = glob(os.path.join(new_path + '/*.jpg'))
print(len(new_filenames))

1377


In [4]:
print(new_filenames[0])
print(new_filenames[0].split('/')[-1])
print(new_filenames[0].split('/')[-1].split('_')[0])

/home/xin/Project/2024 full image dataset_3H7C_corrected/CLLDiKa_20240424_1to30_40min_1_09.jpg
CLLDiKa_20240424_1to30_40min_1_09.jpg
CLLDiKa


In [6]:
img = cv2.imread(new_filenames[0])
print(img.shape)

(1002, 1004, 3)


Segment each image into 64 patches, using sliding-window method with 50% overlapping.

Patch size = 224

image size = 1004*1002

padding = left 3; right 3; up 2; down 2

In [6]:
patch_size = 224

In [7]:
# modify this directory according to your data path

def create_patches(image_name, outdir):
  """
  this function creats patches and stores patch images
  the processing is implemented as follows:
  1) generate the multiplied images
  2) generate the contour for each patch
  3) remove contours that are too small (noise) or too large (not single cell)
  4) find dimensions needed to be zero-padded for each patch
  5) zero padding each selected region to 100*100
  6) resize all the patches to 224*224, according to the input size of the pretrained classification model
  7) save the patches with name: "[IMAGE NAME]_p[ID].jpg"
  """
  counts = 0

  image = cv2.imread(image_name, 0)
  image_width = 1004
  image_height = 1002
  patch_size = 224
 

  for col in range(5):

      if col == 4:
        x_cord = image_width - patch_size
      else:
        x_cord = int(col*patch_size)

      for row in range(5):

        if row == 4:
          y_cord = image_height - patch_size
        else:
          y_cord = int(row*patch_size)

        temp_patch = image[y_cord:y_cord + patch_size, x_cord:x_cord + patch_size]

        patch_num = col*5 + row + 1
        
        patch_filename = f'{image_name.split("/")[-1][:-4]}_p{patch_num}.jpg'
        full_patch_path = os.path.join(outdir, patch_filename)
        #print("Saving patch to:", full_patch_path)
        cv2.imwrite(full_patch_path, temp_patch)
        #cv2.imwrite(outdir + image_name.split('/')[-1][:-4] + '_p'+ str(patch_num) + '.jpg', temp_patch)

In [9]:
class CFG:

    raw_path = '/home/xin/Project/2024 full image dataset_3H7C_corrected'
    patch_out_path = '/home/xin/Project/20240620 stiffness folder_patches for HvsCLL_3H7CLL/Hard'


In [9]:
filenames = (glob(os.path.join(CFG.raw_path + '/*.jpg')))
print(len(filenames))

1377


In [10]:
# test the function using a single image
# plot the figures you want to see the difference before and after multiply the imageJ mask; and the patches generated
create_patches(filenames[12], CFG.patch_out_path)

In [11]:
# # parallel processing
# # modify the [filenames] and [total] if needed
res2 = Parallel(n_jobs=15, backend='threading')(delayed(
     create_patches)(i, CFG.patch_out_path) for i in tqdm(filenames, total=len(filenames)))

  0%|          | 0/1377 [00:00<?, ?it/s]

Get filenames for all generated patches

In [10]:
# check the number of output patches to see if processing was successful
# 248*64 = 15872
total_patch = glob(os.path.join(CFG.patch_out_path + '/*.jpg'))
print(len(total_patch))

12025


Copy and Paste to designated directory

In [11]:
def copyfiles(img, outdir):
    shutil.copyfile(img, outdir)

In [12]:
def split_donor(filenames, outdir):
#   """
#   this function splits the patch images by stiffness according to their names
#   """
  num = len(filenames)
#   donor_name = []



  for i in range(num):
    # print('fn',filenames[i])
    curr_name = filenames[i].split('/')[-1]
    # print('cn',curr_name)
    donor = curr_name.split('_')[0]
    
    #print('st',stiffness)
    if donor[0] == 'C':
      dst = outdir +  '/1/' + curr_name
      copyfiles(filenames[i], dst)
    #elif stiffness == '1to50':
      #dst = outdir +  '/Soft/' + curr_name
      #copyfiles(filenames[i], dst)
    else:
      dst = outdir +  '/0/' + curr_name
      copyfiles(filenames[i], dst)

In [13]:
outdir = '/home/xin/Project/20240620 Hard only_patches for HvsCLL_3H7CLL'
split_donor(total_patch, outdir)

In [14]:
!pip install split-folders

/bin/bash: /home/jia/anaconda3/envs/xin/lib/python3.10/site-packages/cv2/../../../../lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [20]:
import splitfolders
splitfolders.ratio('/home/xin/Project/20240620 Hard only_patches for HvsCLL_3H7CLL', output='/home/xin/Project/20240620 balanced folder for HvsCLL_Hard only', seed=1337, ratio=(0.8,0.1,0.1))

Copying files: 8137 files [00:01, 7321.48 files/s]


In [19]:
import os
import random
import shutil

def randomly_select_images(source_folder, destination_folder):
    # Create the destination folder if it doesn't exist
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    # List all files in the source folder
    all_files = os.listdir(source_folder)

    # Randomly shuffle the list of files
    random.shuffle(all_files)

    # Determine the number of files to copy (half of the total number)
    num_files_to_copy = len(all_files) // 2

    # Select random files and copy them to the destination folder
    for i in range(num_files_to_copy):
        selected_file = all_files[i]
        source_file_path = os.path.join(source_folder, selected_file)
        destination_file_path = os.path.join(destination_folder, selected_file)
        shutil.copy(source_file_path, destination_file_path)

# Example usage:
source_folder = "/home/xin/Project/20240620 Hard only_patches for HvsCLL_3H7CLL/1"
destination_folder = "/home/xin/Project/20240620 Hard only_patches for HvsCLL_3H7CLL/half images of 1"
randomly_select_images(source_folder, destination_folder)